# Can we predict how much a dam removal changes water quality?

**Question:** using only things known *before* a dam is removed (dam height, river size, land use, climate), can we
predict how much water quality changes downstream afterwards?

**How the analysis works.** It runs in stages, with a check after each one. The full plan is
`docs/predictive_feasibility_prespec.md`.

| Stage | Plain meaning | ML term | Sections |
|---|---|---|---|
| 1. Audit | Which measures, dams, regions and predictors have enough data? | EDA | 1–3 |
| 2. Targets | One "how much did it change" number per dam, with an error bar | preprocessing / label construction | 4–7 |
| 3. Cross-validation | Do dam traits predict that number for dams in regions the model has never seen? | modeling + evaluation | 9 |

**How to read it**
- **Target** = one dam's change in a measure after removal: downstream stations within 10 km, adjusted for season
  and weather, standardized to 5 km downstream, averaged over years 1–5 after removal.
  - Conductance, phosphorus and turbidity are on a **log10 scale**: 0.1 ≈ +26%, −0.1 ≈ −21%.
- **Threshold** = the "too small to matter" size from the main analysis (e.g. 0.5 °C, or a 25% change in turbidity).
  It's drawn on the charts as dashed lines.
- **Prediction is not causation.** "Taller dams predict bigger changes" would be an association across dams, not
  proof that height causes it.
- **"Not predictable" is an acceptable answer,** and the main results make it the expected one.

**Read-only.** This notebook reads the CSVs written by `analysis/predictive_feasibility.py`
(`STAGE=audit`, `targets`, `cv` with `data/slurm/predictive_feasibility.slurm`) and fits nothing.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Find the project folder and the predictive outputs written by the cluster jobs.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import predictive_feasibility as pf
from analysis.stage4_report import SESOI, sesoi_label

CASE_STUDY = "all_removals"
OUT = Path(os.environ.get("DAM_REMOVAL_PREDICTIVE_DIR", pf.output_dir_for(CASE_STUDY, ROOT)))
IDS = {"AR_ID": str, "huc2": str, "huc4": str, "fold": str}


def read(name):
    path = OUT / f"{name}.csv"
    if not path.exists() or path.stat().st_size <= 1:
        return pd.DataFrame()
    return pd.read_csv(path, dtype=IDS)


def as_bool(s):
    return s.map({True: True, False: False, "True": True, "False": False}).fillna(False).astype(bool)


MEASURES = pf.TARGET_SERIES
NAMES = {"water_temp": "Water temperature", "dissolved_oxygen": "Dissolved oxygen", "ph": "pH",
         "specific_conductance": "Specific conductance", "phosphorus (Total)": "Total phosphorus",
         "turbidity": "Turbidity", "tds": "TDS", "ssc": "SSC", "tss": "TSS",
         "orthophosphate (Dissolved)": "Orthophosphate"}
LOG = {s for s in NAMES if s.split(" (")[0] in mm.SETTINGS["log_parameters"]}


def unit(s):
    return "log10 change (0.1 ≈ +26%)" if s in LOG else mm.unit_label(s, False)


def heatmap(table, ax, fmt="{:.0f}", cmap="Blues", vmin=None, vmax=None, title=""):
    a = table.to_numpy(dtype=float)
    ax.imshow(a, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            if np.isfinite(a[i, j]):
                ax.text(j, i, fmt.format(a[i, j]), ha="center", va="center", fontsize=8)
    ax.set_xticks(range(a.shape[1]), [str(c) for c in table.columns], rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(a.shape[0]), [str(r) for r in table.index], fontsize=8)
    ax.set_title(title, fontsize=10)


def grid(n=len(MEASURES), cols=3, w=5, h=3.4):
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(w * cols, h * rows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


audit = {n: read(f"audit_{n}") for n in
         ["measures", "per_dam", "regions", "features", "predictors", "pairs", "vif", "predictors_by_region",
          "distance", "distance_by_region", "temporal", "temporal_by_year", "recommendations", "frozen"]}
corr = read("audit_correlations")
targets = read("targets")
if len(targets):
    targets["weak_support"] = as_bool(targets["weak_support"])
stations, comparison, reliab, std = (read("targets_stations"), read("targets_comparison"),
                                     read("targets_reliability"), read("targets_standardization"))
cv = {n: read(f"cv_{n}") for n in ["sizes", "predictions", "metrics", "folds", "huc4_repeats", "decisions",
                                   "selection", "influence_targets", "influence_r2"]}
prim = targets[targets["version"].eq("primary_10km")] if len(targets) else targets

print(f"Outputs: {OUT}")
for stage, name in [("1 audit", "report_audit.md"), ("2 targets", "report_targets.md"), ("3 cv", "report_cv.md")]:
    p = OUT / name
    when = pd.Timestamp(p.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M") if p.exists() else "not run yet"
    print(f"  Stage {stage:10s} {when}")

## 1. Which measures have enough data? (Stage 1 audit)

A measure goes on to modeling only if it has:
- **≥ 60 dams** with a target (about 15 dams per predictor), and
- **≥ 5 regions (HUC2) with ≥ 5 dams each.** The main test holds out one region at a time, so each region needs
  enough dams to be a real test.

"Estimable" = samples in at least 2 different years both before and after removal.

In [ ]:
m = audit["measures"]
if m.empty:
    print("No audit yet: sbatch data/slurm/predictive_feasibility.slurm")
else:
    m10 = m[m["window"].eq("10 km")].copy()
    m10["measure"] = m10["series"].map(NAMES).fillna(m10["series"])
    m10["goes to modeling"] = m10["series"].isin(MEASURES)
    display(m10[["measure", "dams", "stations", "daily_values", "dams_estimable", "huc2_regions",
                 "huc2_regions_ge5_dams", "dams_one_station", "meets_size_rule", "goes to modeling"]]
            .sort_values("dams_estimable", ascending=False).reset_index(drop=True))

In [ ]:
r = audit["regions"]
if len(r):
    t = (r[r["series"].isin(MEASURES)].pivot_table(index="series", columns="huc2", values="dams")
         .reindex(MEASURES).rename(index=NAMES))
    t = t[t.sum().sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(13, 3.2))
    heatmap(t, ax, title="Dams per region (HUC2) and measure; regions with < 5 dams are pooled into one test fold")
    plt.tight_layout()
    plt.show()

**What to look for:** a few regions hold most of the dams. When one of the big regions is held out, the model trains
on fewer dams and has to predict a region that may be quite different. That makes this test harder than a random
split, on purpose: it's the realistic question ("a removal somewhere new").

## 2. The predictors

The candidates, in priority order (`docs/predictive_feasibility_prespec.md` §5). All are known before removal:
dam height, dam drainage area, % agriculture / forest / developed upstream of the downstream stations, pre-removal
air temperature and precipitation, dam age, mapped reservoir area.

**Rules, fixed before any model was fit:**
- A predictor is dropped if it is **< 70% complete.**
- Of two predictors with |correlation| > 0.8, the lower-priority one is dropped.
- At most **1 predictor per 15 dams.** A predictor with ≥ 10% missing also needs a "was missing" column, and that
  column counts toward the limit.
- The reservoir-volume proxy is dropped (Amendment 1). It is built from height, slope and drainage area, so it
  mostly repeats them.

In [ ]:
p = audit["predictors"]
if len(p):
    t = (p[p["series"].isin(MEASURES)].pivot_table(index="predictor", columns="series", values="completeness")
         .reindex([c for c in list(pf.PREDICTORS) + list(pf.SENSITIVITY_FEATURES) if c in set(p["predictor"])])
         .rename(columns=NAMES) * 100)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    heatmap(t, ax, fmt="{:.0f}%", cmap="RdYlGn", vmin=0, vmax=100,
            title="Predictor completeness among each measure's dams (rule: ≥ 70%)")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(corr):
    c = corr.set_index(corr.columns[0])
    c = c.loc[[i for i in pf.PREDICTORS if i in c.index], [i for i in pf.PREDICTORS if i in c.columns]]
    fig, ax = plt.subplots(figsize=(7.5, 6))
    heatmap(c, ax, fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1,
            title="Spearman correlation between predictors (rule: drop one of a pair above |0.8|)")
    plt.tight_layout()
    plt.show()
if len(audit["vif"]):
    print("Variance inflation (how much each predictor overlaps the others; above ~5 = a lot):")
    display(audit["vif"].pivot_table(index="predictor", columns="series", values="vif").rename(columns=NAMES).round(2))

In [ ]:
f, pd_ = audit["features"], audit["per_dam"]
if len(f) and len(pd_):
    est = pd_.loc[as_bool(pd_["estimable"]) & pd_["series"].isin(MEASURES), "AR_ID"].unique()
    f = f[f["AR_ID"].isin(est)]
    names = [n for n in pf.PREDICTORS if n in f]
    fig, axes = grid(len(names), cols=4, w=3.6, h=2.6)
    for ax, n in zip(axes, names):
        v = f[n].dropna()
        ax.hist(v, bins=25, color="#5d6d7e")
        ax.set_title(f"{n}\n{len(v)} of {len(f)} dams ({len(v) / len(f):.0%})", fontsize=9)
        ax.tick_params(labelsize=7)
    plt.suptitle("Predictor distributions (dams with a target in any measure)", y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
pr = audit["predictors_by_region"]
if len(pr):
    cols = [c for c in pr if c.endswith("_missing")]
    t = pr.set_index("huc2")[cols].rename(columns=lambda c: c.replace("_missing", "")).T * 100
    fig, ax = plt.subplots(figsize=(13, 3.8))
    heatmap(t, ax, fmt="{:.0f}", cmap="Oranges", vmin=0, vmax=100, title="% missing, by region (HUC2)")
    plt.tight_layout()
    plt.show()

**Why missingness by region matters:** if a predictor is missing for nearly a whole region, a model tested on that
region falls back on the "was missing" column and the median, so it predicts little there. Imputation and scaling are
fitted on the training regions only, so the held-out region can't leak into them.

## 3. Distance and time coverage

**The question here:** could each dam get its own curve over distance or time? No:
- **Distance:** 74% of dams have a single station within 10 km, so per-dam distance curves can't be estimated.
  Instead every dam is standardized to 5 km with the frozen distance model's slope.
- **Time:** few dams have enough years to estimate their own time profile. The target is the average over years 1–5,
  with years 1–3 as a sensitivity check.

In [ ]:
if len(audit["distance"]):
    d = audit["distance"]
    d = d[d["series"].isin(MEASURES)].set_index("series").rename(index=NAMES)
    display(d[["station_km_median", "share_stations_le_2km", "dams_le_10km", "dams_ge2_stations_le_10km",
               "dams_span_ge_3km", "dams_all_3_coverage_bands", "dams_only_beyond_10km"]].round(2))
p10 = stations[stations["version"].eq("primary_10km")] if len(stations) else stations
if len(p10):
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
    for s in MEASURES:
        km = p10.loc[p10["series"].eq(s), "km"].dropna()
        axes[0].hist(km, bins=np.arange(0, 10.5, 0.5), histtype="step", lw=1.5, label=NAMES[s])
    axes[0].axvline(5, ls="--", c="k", lw=1)
    axes[0].set(xlabel="station distance downstream (km)", ylabel="stations",
                title="Where the downstream stations are (dashed: 5 km standardization point)")
    axes[0].legend(fontsize=7)
    ty = audit["temporal_by_year"]
    if len(ty):
        for s in MEASURES:
            g = ty[ty["series"].eq(s)].sort_values("years_from_removal")
            axes[1].plot(g["years_from_removal"], g["dams"], marker="o", ms=3, label=NAMES[s])
        axes[1].axvline(0, c="grey", lw=8, alpha=0.3)
        axes[1].set(xlabel="years from removal (removal year left out)", ylabel="dams with data",
                    title="Dams with data, year by year")
    plt.tight_layout()
    plt.show()

## 4. The targets: one change per dam (Stage 2)

For each measure, one regression on the daily values (the frozen model's rows):

$$y = \text{station baseline} + \text{month} + \text{weather} + \delta_{\text{station}}\cdot\text{after}$$

- **Each station gets its own before/after change δ.** Nothing is pooled across dams, so one dam's target doesn't
  borrow from others, unlike the main model's dam estimates (BLUPs).
- **A dam's target** = the average of its stations' δ (each station counts equally), shifted to what it would be at
  5 km downstream and a reference dilution, using the frozen distance model.
- **Error bar:** allows for values in the same station-year being correlated (cluster-robust "CR2"). It also adds the
  uncertainty of the frozen distance slope, conservatively.
- **"Weak support"** flags a dam if any of these holds:
  - fewer than 6 station-years
  - fewer than 20 daily values
  - an error bar over 3× the measure's median
  - the error bar disagrees with a simpler year-means version by more than 2×

In [ ]:
if prim.empty:
    print("No targets yet: STAGE=targets sbatch data/slurm/predictive_feasibility.slurm")
else:
    rows = []
    for s in MEASURES:
        t = prim[prim["series"].eq(s)]
        thr = SESOI.get(s, np.nan)
        rows.append({"measure": NAMES[s], "dams": len(t), "regions": t["huc2"].nunique(),
                     "threshold": sesoi_label(s), "median change": t["target"].median(),
                     "SD between dams": t["target"].std(),
                     "% dams beyond threshold": (t["target"].abs() > thr).mean() * 100,
                     "% whose 95% range excludes 0": ((t["ci_low"] > 0) | (t["ci_high"] < 0)).mean() * 100,
                     "median error bar (SE)": t["se_total"].median(), "weak support": int(t["weak_support"].sum())})
    display(pd.DataFrame(rows).round(3))

In [ ]:
if len(prim):
    fig, axes = grid()
    for ax, s in zip(axes, MEASURES):
        t = prim[prim["series"].eq(s)]
        thr = SESOI.get(s, np.nan)
        ax.hist([t.loc[~t["weak_support"], "target"], t.loc[t["weak_support"], "target"]], bins=25,
                stacked=True, color=["#2e86c1", "#f0b27a"], label=["well supported", "weak support"])
        for x in (-thr, thr):
            ax.axvline(x, ls="--", c="k", lw=1)
        ax.axvline(0, c="grey", lw=0.8)
        ax.set_title(f"{NAMES[s]} ({len(t)} dams)", fontsize=10)
        ax.set_xlabel(unit(s), fontsize=8)
    axes[0].legend(fontsize=7)
    plt.suptitle("How much each dam changed (dashed = the 'too small to matter' threshold)", y=1.01)
    plt.tight_layout()
    plt.show()

**What to look for:** the middle sits near 0 (the *average* dam barely changes, matching the main results), but the
spread is wide. That spread is what a predictive model would have to explain.

In [ ]:
if len(prim):
    fig, axes = grid(h=3.8)
    for ax, s in zip(axes, MEASURES):
        t = prim[prim["series"].eq(s)].sort_values("target").reset_index(drop=True)
        c = np.where(t["weak_support"], "#e67e22", "#2e86c1")
        ax.vlines(t.index, t["ci_low"], t["ci_high"], colors=c, lw=0.8, alpha=0.6)
        ax.scatter(t.index, t["target"], c=c, s=6, zorder=3)
        thr = SESOI.get(s, np.nan)
        ax.axhspan(-thr, thr, color="grey", alpha=0.15)
        ax.axhline(0, c="k", lw=0.6)
        ax.set_title(NAMES[s], fontsize=10)
        ax.set_xlabel("dams, sorted by change", fontsize=8)
        ax.set_ylabel(unit(s), fontsize=8)
    plt.suptitle("Each dam's change with its 95% range (orange = weak support; grey band = within the threshold)",
                 y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
if len(prim):
    flags = [c for c in prim if c.startswith("flag_")]
    t = (prim[flags + ["weak_support"]].apply(as_bool).groupby(prim["series"]).sum()
         .reindex(MEASURES).rename(index=NAMES))
    t.columns = [c.replace("flag_", "") for c in t.columns]
    print("Dams flagged for weak support, by reason (a dam can have several):")
    display(t.astype("Int64"))

## 5. Are the differences between dams real, or noise? (reliability)

**Reliability** = the share of the dam-to-dam spread that is real rather than measurement noise:
τ² / (τ² + average error²), where τ² is the real between-dam variance.
- **Near 1:** the targets are precise.
- **Near 0:** the differences are mostly noise, and nothing could predict them.
- **Rule:** a measure stops here if every estimate is below **0.15**.

τ² is estimated four ways:
- **two from the targets themselves** (Paule–Mandel, DerSimonian–Laird), which subtract each dam's own error first
- **two from the frozen models** (the 10 km before/after model and the 30 km distance model)

**What it means for prediction:** even a perfect model could explain at most about this share of the observed spread.
It is a rough guide, not an exact ceiling.

In [ ]:
if len(reliab):
    r = reliab[reliab["version"].eq("primary_10km") & reliab["se_kind"].eq("se_total")]
    t = r.pivot_table(index="series", columns="tau2_source", values="reliability").reindex(MEASURES)
    fig, ax = plt.subplots(figsize=(11, 3.6))
    x = np.arange(len(t))
    for i, col in enumerate(t.columns):
        ax.bar(x + (i - 1.5) * 0.2, t[col], width=0.2, label=col)
    ax.axhline(pf.RELIABILITY_STOP, c="#c0392b", ls="--", lw=1.2, label="stop rule (0.15)")
    ax.set_xticks(x, [NAMES[s] for s in t.index], fontsize=9)
    ax.set(ylim=(0, 1), ylabel="reliability", title="Share of the dam-to-dam spread that is real (10 km targets)")
    ax.legend(fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()
    display(r[["series", "tau2_source", "tau", "mean_se2", "reliability", "stop_rule_met"]].round(3)
            .reset_index(drop=True))

## 6. Does the target depend on how it's built?

The same per-dam change, built five other ways:
- without the 5 km standardization
- nearest station only
- years 1–3 only
- the main model's dam estimates (BLUPs, which are shrunk toward the average, so their spread is smaller)
- a 30 km window (more dams)

**A high correlation means the choice of definition doesn't matter much.** Rank correlation (Spearman) is less
sensitive to a few extreme dams than Pearson. A big gap between the two means a handful of dams disagree
(see section 7).

In [ ]:
if len(comparison):
    c = comparison.copy()
    c["measure"] = c["series"].map(NAMES)
    display(c.pivot_table(index="measure", columns="version", values="pearson_r").round(2)
            .style.background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1).format("{:.2f}")
            .set_caption("Pearson correlation with the primary target"))
    display(c.pivot_table(index="measure", columns="version", values="spearman_r").round(2)
            .style.background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1).format("{:.2f}")
            .set_caption("Spearman (rank) correlation with the primary target"))

In [ ]:
versions = ["nearest_station", "years_1_3", "frozen_blup_10km", "sensitivity_30km"]
if len(prim):
    fig, axes = plt.subplots(len(MEASURES), len(versions), figsize=(15, 3 * len(MEASURES)), squeeze=False)
    for i, s in enumerate(MEASURES):
        p = prim[prim["series"].eq(s)][["AR_ID", "target"]]
        for j, v in enumerate(versions):
            ax = axes[i, j]
            alt = targets[targets["version"].eq(v) & targets["series"].eq(s)][["AR_ID", "target"]]
            m = p.merge(alt, on="AR_ID", suffixes=("", "_alt"))
            ax.scatter(m["target"], m["target_alt"], s=8, alpha=0.6)
            lim = np.nanpercentile(np.r_[m["target"], m["target_alt"]], [0.5, 99.5]) if len(m) else [0, 1]
            ax.plot(lim, lim, c="grey", lw=0.8)
            r = m["target"].corr(m["target_alt"]) if len(m) > 2 else np.nan
            ax.set_title(f"{NAMES[s]} vs {v}  (r = {r:.2f}, {len(m)} dams)", fontsize=8)
            ax.tick_params(labelsize=7)
            if j == 0:
                ax.set_ylabel("other version", fontsize=8)
            if i == len(MEASURES) - 1:
                ax.set_xlabel("primary target", fontsize=8)
    plt.tight_layout()
    plt.show()

**What to look for:**
- Points on the diagonal mean the two versions agree.
- **Years 1–3 vs 1–5 for phosphorus and turbidity agree less than the other measures** (r ≈ 0.66 and 0.73).
  - Either their response changes over time, or the later years are noisy.
  - So a prediction for these two means *the average over years 1–5*, not an early or fixed response.
  - Stage 3 repeats the analysis on the years 1–3 targets to check (Amendment 2c).

## 7. Conductance: which dams cause the Pearson / Spearman gap?

**The pattern:** conductance ranks dams almost identically across versions (Spearman ≈ 0.97), but Pearson is lower
(0.73 against the nearest-station version). That points to a few extreme dams.
- The table lists, for each dam, how much the correlation would *rise* if that dam were left out.
- **These dams stay in the analysis** (Amendment 2b) unless a real data problem is found. An extreme but genuine
  response is exactly what a predictive model should try to explain.
- `notebooks/qa/conductance_plausibility.ipynb` checks conductance for unit errors.

In [ ]:
s = "specific_conductance"
if len(targets):
    infl = pf.target_influence(targets, s)
    top = infl.groupby("version").head(5)
    display(top[["version", "AR_ID", "huc2", "target", "target_alt", "stations", "min_km", "max_km", "days",
                 "clusters", "weak_support", "pearson_all", "pearson_without", "r_gain_if_left_out"]]
            .round(3).reset_index(drop=True))
    worst = top["AR_ID"].unique()
    if len(stations):
        st = stations[stations["series"].eq(s) & stations["version"].eq("primary_10km")
                      & stations["AR_ID"].isin(worst)]
        print("Their stations (primary 10 km): change per station, distance, days, station-years")
        display(st[["AR_ID", "station_id", "delta", "km", "dar", "days", "clusters", "pre_years", "post_years",
                    "delta_ym"]].round(3).sort_values(["AR_ID", "km"]).reset_index(drop=True))

In [ ]:
if len(targets):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    p = prim[prim["series"].eq(s)][["AR_ID", "target"]]
    for ax, v in zip(axes, ["nearest_station", "frozen_blup_10km"]):
        alt = targets[targets["version"].eq(v) & targets["series"].eq(s)][["AR_ID", "target"]]
        m = p.merge(alt, on="AR_ID", suffixes=("", "_alt"))
        hit = m["AR_ID"].isin(worst)
        ax.scatter(m.loc[~hit, "target"], m.loc[~hit, "target_alt"], s=10, alpha=0.6)
        ax.scatter(m.loc[hit, "target"], m.loc[hit, "target_alt"], s=30, c="#c0392b")
        for _, r in m[hit].iterrows():
            ax.annotate(r["AR_ID"], (r["target"], r["target_alt"]), fontsize=7)
        ax.set(xlabel="primary target (log10)", ylabel=f"{v} (log10)", title=f"Conductance: primary vs {v}")
    plt.tight_layout()
    plt.show()

## 8. Why this notebook doesn't plot targets against predictors

Plotting "change vs dam height" here would be looking at the answer before the test. It could tempt us, even without
meaning to, to pick predictors or transformations that fit *these* dams, and that would inflate the results.
- **The predictor rules (section 2) were fixed using the features alone,** never the targets.
- **The only honest look at target vs predictor is section 9:** predictions for regions the model never saw.

## 9. Can the targets be predicted? (Stage 3 cross-validation)

**Four models per measure:**
- **A, overall mean:** predicts the average dam for everyone. This is the "no skill" benchmark.
- **B, region mean:** predicts the average dam of the same region.
- **C, ridge regression (the main model):** a linear model on the selected predictors, with shrinkage chosen
  automatically.
- **D, small random forest:** checks for non-linear patterns. Its results are labelled exploratory.

**How they're tested:**
- **Main test:** hold out one whole region (HUC2), train on the rest, predict the held-out dams, and repeat for every
  region.
- **Second test:** groups of nearby watersheds (HUC4) held out, 5 folds, repeated 20 times. Model B can only be
  judged here, because under the main test the held-out region is never in training.
- **Everything is fitted on the training dams only:** predictor choice, filling in missing values, scaling and the
  amount of shrinkage.

**Out-of-sample R²** = how much better than model A: 0 = no better, 0.10 = 10% less squared error, below 0 = worse.

**A measure "passes" (go) only if model C meets all five:**
1. R² ≥ 0.05 in the main test
2. it beats the region mean (B) in the second test
3. it beats A in at least half the held-out regions with ≥ 10 dams
4. a 90% bootstrap range for R² (resampling regions) is above 0
5. its typical error is at least 5% smaller than A's

In [ ]:
dec = cv["decisions"]
if dec.empty:
    print("Stage 3 has not run yet: STAGE=cv sbatch data/slurm/predictive_feasibility.slurm")
else:
    d = dec[dec["analysis"].eq("primary")].copy()
    d["measure"] = d["series"].map(NAMES)
    crit = ["c1_r2_ge_0_05", "c2_beats_region_baseline", "c3_beats_A_half_of_folds", "c4_boot90_excludes_0",
            "c5_mae_5pct_lower"]
    show = d[["measure", "model", "role", "r2_loro", "r2_huc4", "r2_huc4_B", "share_folds_beating_A",
              "r2_boot_q05", "mae_vs_A"] + crit + ["go"]].copy()
    for c in crit + ["go"]:
        show[c] = as_bool(show[c]).map({True: "✓", False: "✗"})
    display(show.round(3).reset_index(drop=True))
    for _, r in d[d["model"].eq("C")].iterrows():
        verdict = ("**predictable** (passes all five)" if as_bool(pd.Series([r["go"]])).iloc[0]
                   else "**not reliably predictable**")
        display(Markdown(f"- {NAMES[r['series']]}: {verdict}; out-of-sample R² = {r['r2_loro']:.3f}"))

In [ ]:
met = cv["metrics"]
if len(met):
    m = met[met["analysis"].eq("primary")]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    x = np.arange(len(MEASURES))
    lo = m[m["scheme"].eq("loro")]
    for i, mod in enumerate(["C", "D"]):
        g = lo[lo["model"].eq(mod)].set_index("series").reindex(MEASURES)
        err = np.vstack([g["r2"] - g["r2_boot_q05"], g["r2_boot_q95"] - g["r2"]])
        axes[0].bar(x + (i - 0.5) * 0.35, g["r2"], width=0.35, yerr=err, capsize=3,
                    label=f"{mod} ({'ridge' if mod == 'C' else 'forest, exploratory'})")
    axes[0].set_title("Main test (hold out one region): R² with 90% bootstrap range")
    h4 = m[m["scheme"].eq("huc4_repeated")]
    for i, mod in enumerate(["B", "C", "D"]):
        g = h4[h4["model"].eq(mod)].set_index("series").reindex(MEASURES)
        axes[1].bar(x + (i - 1) * 0.27, g["r2"], width=0.27, yerr=g["r2_sd_repeats"], capsize=3,
                    label={"B": "B region mean", "C": "C ridge", "D": "D forest"}[mod])
    axes[1].set_title("Second test (hold out HUC4 groups, 20 repeats): mean R² ± SD")
    for ax in axes:
        ax.axhline(0, c="k", lw=0.8)
        ax.axhline(pf.R2_MIN, c="#c0392b", ls="--", lw=1, label="0.05 (criterion 1)")
        ax.set_xticks(x, [NAMES[s] for s in MEASURES], fontsize=8, rotation=15)
        ax.set_ylabel("out-of-sample R² (vs overall mean)")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    display(m[["series", "scheme", "model", "dams", "r2", "mae", "mae_natural", "mae_vs_A", "rmse",
               "direction_accuracy", "auc_beyond_sesoi"]].round(3).reset_index(drop=True))

**Reading the table:**
- **mae_natural:** the typical prediction error in natural units (a % change for log measures). Compare it with the
  threshold.
- **mae_vs_A:** below 0.95 means at least 5% better than predicting the average.
- **direction_accuracy:** the share of dams where the model got the sign (up or down) right.
- **AUC:** how well |prediction| separates dams beyond the threshold from those within it (0.5 = chance). It is
  shown only when both groups have ≥ 15 dams.

In [ ]:
pred = cv["predictions"]
if len(pred):
    p = pred[pred["analysis"].eq("primary")]
    fig, axes = grid()
    for ax, s in zip(axes, MEASURES):
        g = p[p["series"].eq(s)]
        ax.scatter(g["pred_C"], g["y"], s=8, alpha=0.6)
        lim = np.nanpercentile(np.r_[g["y"], g["pred_C"]], [1, 99]) if len(g) else [0, 1]
        ax.plot(lim, lim, c="grey", lw=0.8)
        ax.axhline(0, c="k", lw=0.4)
        ax.set_title(f"{NAMES[s]}: observed vs predicted (ridge)", fontsize=9)
        ax.set_xlabel(f"predicted ({unit(s)})", fontsize=8)
        ax.set_ylabel("observed", fontsize=8)
    plt.suptitle("Predictions for dams in held-out regions; no skill = a vertical cloud", y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
fo = cv["folds"]
if len(fo):
    f = fo[fo["analysis"].eq("primary")].copy()
    f["C vs A"] = f["mse_C"] / f["mse_A"]
    t = f.pivot_table(index="series", columns="fold", values="C vs A").reindex(MEASURES).rename(index=NAMES)
    n = f.pivot_table(index="series", columns="fold", values="dams").reindex(MEASURES)
    fig, ax = plt.subplots(figsize=(13, 3.4))
    heatmap(t, ax, fmt="{:.2f}", cmap="RdYlGn_r", vmin=0.6, vmax=1.4,
            title="Held-out region: ridge error ÷ mean-only error (< 1 = ridge better)")
    plt.tight_layout()
    plt.show()
    print("Dams per held-out region:")
    display(n.rename(index=NAMES).fillna(0).astype(int))

### Sensitivity analyses

These are checks, not new decisions: the go/no-go above comes from the primary analysis only.

| Analysis | What changes |
|---|---|
| weighted | precise dams count more: weight 1 / (τ² + SE²) |
| complete_cases | only dams with no missing predictors |
| removal_year | removal year added as a predictor (it mixes real trends with changes in monitoring) |
| exclude_weak_support | weak-support dams left out (Amendment 2a) |
| years_1_3 | targets from years 1–3 only. **The prespecified comparison is phosphorus and turbidity** (Amendment 2c) |
| window_30km | the 30 km targets (about twice the dams) |

In [ ]:
if len(dec):
    c = dec[dec["model"].eq("C")]
    order = [a for a in pf.CV_ANALYSES if a in set(c["analysis"])]
    t = c.pivot_table(index="series", columns="analysis", values="r2_loro").reindex(MEASURES)[order]
    fig, ax = plt.subplots(figsize=(11, 3.4))
    heatmap(t.rename(index=NAMES), ax, fmt="{:+.3f}", cmap="RdYlGn", vmin=-0.15, vmax=0.15,
            title="Ridge R², main test, per analysis")
    plt.tight_layout()
    plt.show()
    g = c.assign(go=as_bool(c["go"]).map({True: "go", False: "—"}))
    display(g.pivot_table(index="series", columns="analysis", values="go", aggfunc="first")
            .reindex(MEASURES)[order].rename(index=NAMES))
    display(cv["sizes"].pivot_table(index="series", columns="analysis", values="dams").reindex(MEASURES)[order]
            .rename(index=NAMES).astype("Int64"))

In [ ]:
sel = cv["selection"]
if len(sel):
    t = (sel[sel["analysis"].eq("primary")].pivot_table(index="column", columns="series", values="share_of_folds")
         .reindex(columns=MEASURES).rename(columns=NAMES) * 100)
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(t) + 1.5))
    heatmap(t, ax, fmt="{:.0f}%", cmap="Purples", vmin=0, vmax=100,
            title="Which predictors the ridge model used (% of held-out-region folds)")
    plt.tight_layout()
    plt.show()

In [ ]:
ir = cv["influence_r2"]
if len(ir):
    print("Conductance: how much each dam moves the ridge R² (main test) if left out; top 10 (Amendment 2b)")
    display(ir.head(10).round(4).reset_index(drop=True))
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(ir["r2_change_if_left_out"], bins=40, color="#5d6d7e")
    ax.axvline(0, c="k", lw=0.8)
    ax.set(xlabel="change in R² when the dam is left out", ylabel="dams",
           title=f"Conductance, R² with all dams = {ir['r2_C_all'].iloc[0]:.3f}")
    plt.tight_layout()
    plt.show()

## 10. Limitations to carry into the write-up

- **Observational:** a predictor that works shows association across dams, not cause.
- **Few regions:** the main test has about 6–8 folds, so the bootstrap range is approximate.
- **One number per dam:** the target averages over stations within 10 km and over years 1–5. It can't say *where* or
  *when* a dam's change happens.
- **The standardization to 5 km** uses one slope shared by all dams (the frozen distance model). It barely changes the
  targets (section 6), but it assumes every dam's change fades with distance the same way.
- **Predictors are coarse:** no sediment volume, no removal method, and no reservoir size for most dams.
- **Multiple comparisons:** 6 measures are tested, so a single measure passing is reported cautiously.